# Expérimentations préliminaires — 1. Régression logistique

Régression logistique **standard** (maximum de vraisemblance, sans pénalité ni
hyperparamètre) : $P(\text{victoire locale}) = \sigma(\beta_0 + \boldsymbol\beta^\top \mathbf{z})$.

Pipeline : imputation (médiane) → standardisation → **ACP (critère de Kaiser, λ > 1)** →
régression logistique. Pour chaque pli, tout le pipeline est ajusté sur les saisons
d'entraînement, puis évalué sur la saison testée. Rien n'est ajusté sur la saison testée.

**Pourquoi l'ACP ?** Les 1 052 variables sont très redondantes (voir
`00_donnees_protocole`). L'ACP retire cette colinéarité, qui rend les coefficients
instables, et réduit fortement le nombre de paramètres à estimer.

**Combien de composantes ?** Critère de Kaiser : on garde les composantes dont la valeur
propre est supérieure à 1. Les variables étant standardisées (variance 1 chacune), une
composante retenue résume plus de variance qu'une variable d'origine. Le nombre de
composantes est déterminé sur l'entraînement de chaque pli.

**Protocole commun** (voir `src/nhl_forecast/features/equipes.py` et
`src/nhl_forecast/evaluation/preliminaire.py`) :

- saisons régulières 2020-21 à 2025-26, statistiques d'**équipes** seulement, **toutes les
  colonnes** de `t_summary`, `t_percentages`, `t_summaryshooting`, `t_outshootoutshotby` et
  MoneyPuck (5 situations de jeu) ; seuls les identifiants, les colonnes vides ou constantes et
  les doublons exacts sont retirés, automatiquement ;
- validation croisée **chronologique** à 5 plis (fenêtre croissante) : le pli *k* entraîne sur
  les saisons 1 à *k* et teste sur la saison *k* + 1 ;
- hyperparamètres choisis sur les 20 % les plus récents des matchs d'entraînement du pli ;
- métriques : log loss, score de Brier, accuracy (seuil 0,5).

Avec `SAUVER = True`, les tableaux sont enregistrés en CSV (séparateur « ; ») dans
`reports/map6009/tables/` (préfixe `exp_`) et les prédictions dans
`data/processed/experimentation/` (lues par `04_comparaison.ipynb`).

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from nhl_forecast.evaluation.preliminaire import (
    GRAINE,
    ACPKaiser,
    exporter_tableau,
    formater,
    plis_chronologiques,
    predictions,
    resultats_par_pli,
    resume,
    sauver_predictions,
    tableau_plis,
)
from nhl_forecast.features.equipes import (
    NOM_SAISON,
    charger_equipes,
    construire_matchs,
    variables,
)

SAUVER = True
NOM = "Régression logistique"

In [2]:
matchs = construire_matchs(charger_equipes())
X_cols = variables(matchs)
X, y = matchs[X_cols], matchs["victoire_locale"].to_numpy()
plis = plis_chronologiques(matchs)
print(f"{len(matchs)} matchs, {len(X_cols)} variables")

7428 matchs, 1052 variables


C:\Users\tom_9\Projets\nhl-probabilistic-forecasting\src\nhl_forecast\features\equipes.py:182: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matchs = matchs.astype({"visiteur": "int64"}).reset_index()


## 1. Données en entrée

Une ligne par match. Pour chacune des 525 statistiques d'équipe (cinq sources), on calcule
pour chaque équipe, **avant** le match :

- `_saison` : la moyenne de ses matchs précédents de la saison ;
- `_10` : la moyenne de ses 10 derniers matchs (au moins 3, sinon manquant).

La variable du match est l'**écart local − visiteur** de ces moyennes (préfixe `d_`).
S'ajoutent l'écart de jours de repos et de matchs joués. Total : 525 × 2 + 2 = 1 052
variables. Le match lui-même n'entre jamais dans ses propres variables. Exemple pour
une statistique :

In [3]:
ex = [c for c in X_cols if c.startswith("d_mp_all__xGoalsPercentage")] + ["d_repos", "d_matchs_joues"]
formater(matchs[["date", "local_nom", "visiteur_nom", *ex, "victoire_locale"]].dropna().tail(5), 3)

date,local_nom,visiteur_nom,d_mp_all__xGoalsPercentage_saison,d_mp_all__xGoalsPercentage_10,d_repos,d_matchs_joues,victoire_locale
2026-04-16 00:00:00,Winnipeg Jets,San Jose Sharks,0.015,0.082,1.000,0,0
2026-04-16 00:00:00,Utah Mammoth,St. Louis Blues,0.032,-0.026,0.000,0,0
2026-04-16 00:00:00,Calgary Flames,Los Angeles Kings,-0.039,-0.128,0.000,0,1
2026-04-16 00:00:00,Edmonton Oilers,Vancouver Canucks,0.067,0.079,1.000,0,1
2026-04-16 00:00:00,Colorado Avalanche,Seattle Kraken,0.098,0.098,1.000,0,1


## 2. Plis chronologiques

Les six saisons 2020-21 à 2025-26 sont toutes utilisées, mais 2020-21 ne sert qu'à l'entraînement
Chaque pli ajoute une saison à l'entraînement et teste la suivante, ce qui donne 5 saisons testées
(2021-22 à 2025-26).

In [4]:
formater(tableau_plis(matchs))

Pli,Saisons d'entraînement,Saison testée,Matchs (entraînement),dont validation interne,Matchs (test)
1,2020-21 à 2020-21,2021-22,868,174,1312
2,2020-21 à 2021-22,2022-23,2180,436,1312
3,2020-21 à 2022-23,2023-24,3492,698,1312
4,2020-21 à 2023-24,2024-25,4804,961,1312
5,2020-21 à 2024-25,2025-26,6116,1223,1312


## 3. Entraînement et résultats par pli (saison testée)

In [5]:
def pipeline():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         ACPKaiser(random_state=GRAINE),
                         LogisticRegression(max_iter=5000))  # fmt: skip


preds, acp = [], []
for pli in plis:
    m = pipeline().fit(X.iloc[pli["entr"]], y[pli["entr"]])
    preds.append(predictions(matchs, pli, m.predict_proba(X.iloc[pli["test"]])[:, 1], NOM))
    acp.append({"Pli": pli["pli"], "Saison testée": NOM_SAISON[pli["saison_test"]],
                "Composantes ACP": m[-2].n_components_,
                "Variance expliquée (%)": round(100 * m[-2].variance_expliquee_, 1)})
preds = pd.concat(preds, ignore_index=True)
tab_acp = pd.DataFrame(acp)
tab_plis = resultats_par_pli(preds)
if SAUVER:
    exporter_tableau(tab_plis, "exp_reglog_par_pli")
    exporter_tableau(tab_acp, "exp_reglog_composantes")
    sauver_predictions(preds, "01_reglog")
formater(tab_acp)

Pli,Saison testée,Composantes ACP,Variance expliquée (%)
1,2021-22,111,95.3000
2,2022-23,133,92.8000
3,2023-24,140,91.8000
4,2024-25,145,91.3000
5,2025-26,149,90.9000


In [6]:
formater(tab_plis)

Modèle,Pli,Saison testée,n,Log loss,Brier,Accuracy
Régression logistique,1,2021-22,1312,0.7306,0.2597,0.5671
Régression logistique,2,2022-23,1312,0.7159,0.2542,0.5846
Régression logistique,3,2023-24,1312,0.6918,0.2481,0.5686
Régression logistique,4,2024-25,1312,0.6848,0.2455,0.5595
Régression logistique,5,2025-26,1312,0.6944,0.2502,0.5236


## 4. Résumé (moyenne ± écart-type sur les 5 saisons testées)

In [7]:
tab_resume = resume(preds)
if SAUVER:
    exporter_tableau(tab_resume, "exp_reglog_resume")
formater(tab_resume)

Modèle,Log loss,Brier,Accuracy
Régression logistique,0.7035 ± 0.0191,0.2516 ± 0.0056,0.5607 ± 0.0226
